# Notebook 04 — GraphMamba Predicted vs Observed NO₂
## LA Basin Sites · July – September 2024

Compares hourly GraphMamba NO₂ predictions against co-located AirNow observations
for the 21 South Coast AQMD monitoring stations inside the LA Basin bounding box.

**Source:** `/mnt/data3/GraphMamba/output/test/predictions.nc`  
**Variables used:**
- `no2` — GraphMamba predicted NO₂ (PPB)
- `observed_no2` — AirNow in-situ observed NO₂ (PPB)
- `has_observation` — flag: 1 = valid observation, 0 = missing
- `time` — hours since 1970-01-01 (UTC)


In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
from netCDF4 import Dataset, num2date
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import cartopy.crs as ccrs
import cartopy.feature as cfeature

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── Paths ──────────────────────────────────────────────────────────────────────
PRED_NC  = Path('/mnt/data3/GraphMamba/output/test/predictions.nc')
OUT_DIR  = Path('/mnt/data3/isybelle1118/NO2 Forecasting/no2 analysis/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── LA Basin bounding box (same as la_basin_no2_analysis) ──────────────────────
LAT_MIN, LAT_MAX =  33.6,  34.4
LON_MIN, LON_MAX = -118.7, -117.2

# ── Cartopy projections ────────────────────────────────────────────────────────
DATA_CRS = ccrs.PlateCarree()
MAP_PROJ = ccrs.LambertConformal(central_longitude=-118.0, central_latitude=34.0)

# ── Focus sites (subset of LA Basin sites present in predictions.nc) ───────────
FOCUS_SITES  = ['Anaheim', 'Ontario Near Road']
SITE_COLORS  = ['steelblue', 'darkorange']

print('Setup complete.')
print(f'Output directory: {OUT_DIR}')


## 1 · Load & Filter Predictions


In [ ]:
ds = Dataset(PRED_NC)

# ── Decode time ────────────────────────────────────────────────────────────────
raw_t   = ds['time'][:]
nc_dates = num2date(raw_t, ds['time'].units, calendar='proleptic_gregorian')
times   = pd.DatetimeIndex([pd.Timestamp(str(d)) for d in nc_dates])

# ── Site metadata ──────────────────────────────────────────────────────────────
site_names = np.array(ds['site_name'][:])
lats       = ds['latitude'][:].data
lons       = ds['longitude'][:].data
agencies   = np.array(ds['agency'][:])

# ── Filter to LA Basin ─────────────────────────────────────────────────────────
la_mask = (
    (lats >= LAT_MIN) & (lats <= LAT_MAX) &
    (lons >= LON_MIN) & (lons <= LON_MAX)
)
la_idx = np.where(la_mask)[0]

# ── Pull arrays ────────────────────────────────────────────────────────────────
pred_all     = ds['no2'][:, la_idx].data.astype(float)            # (T, N_la)
obs_all      = ds['observed_no2'][:, la_idx].data.astype(float)   # (T, N_la)
has_obs_all  = ds['has_observation'][:, la_idx].data              # (T, N_la)

# Mask invalid values
pred_all[pred_all < 0]    = np.nan
obs_all[has_obs_all != 1] = np.nan   # only keep valid AirNow readings

la_names    = site_names[la_idx]
la_lats     = lats[la_idx]
la_lons     = lons[la_idx]
la_agencies = agencies[la_idx]

ds.close()

print(f'Time steps  : {len(times)}  ({times[0].date()} → {times[-1].date()})')
print(f'LA Basin sites in predictions.nc : {len(la_idx)}')
for i, (n, lat, lon) in enumerate(zip(la_names, la_lats, la_lons)):
    n_valid = int(np.sum(np.isfinite(obs_all[:, i])))
    print(f'  [{i:2d}] {n:<40s}  lat={lat:.4f}  lon={lon:.4f}  obs_valid={n_valid}')


## 2 · LA Basin Sites — Overview Map


In [ ]:
_LA_CITIES = {
    'Los Angeles':    (-118.243, 34.052),
    'Long Beach':     (-118.193, 33.770),
    'Anaheim':        (-117.913, 33.835),
    'Burbank':        (-118.309, 34.181),
    'Pasadena':       (-118.144, 34.148),
    'Ontario':        (-117.648, 34.063),
    'Riverside':      (-117.396, 33.953),
    'San Bernardino': (-117.295, 34.108),
}

def _la_map_axes(fig, spec=111, extent_pad=0.15):
    ax = fig.add_subplot(spec, projection=MAP_PROJ)
    ax.set_extent([LON_MIN - extent_pad, LON_MAX + extent_pad,
                   LAT_MIN - extent_pad, LAT_MAX + extent_pad], crs=DATA_CRS)
    ax.add_feature(cfeature.OCEAN.with_scale('10m'),    facecolor='#c8dff0', zorder=0)
    ax.add_feature(cfeature.LAND.with_scale('10m'),     facecolor='#f5f1eb', zorder=1)
    ax.add_feature(cfeature.LAKES.with_scale('10m'),    facecolor='#c8dff0', zorder=2)
    ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'urban_areas', '10m',
                   facecolor='#ddd5c8', edgecolor='none'), zorder=2)
    ax.add_feature(cfeature.NaturalEarthFeature('physical', 'rivers_lake_centerlines', '10m',
                   facecolor='none', edgecolor='#6699bb', linewidth=0.45), zorder=3)
    ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_2_counties', '10m',
                   facecolor='none', edgecolor='#bbbbbb', linewidth=0.45), zorder=3)
    ax.add_feature(cfeature.NaturalEarthFeature('cultural', 'roads', '10m',
                   facecolor='none', edgecolor='#cc8844', linewidth=0.55), zorder=4)
    ax.add_feature(cfeature.STATES.with_scale('10m'),    edgecolor='#666666', linewidth=0.9, zorder=5)
    ax.add_feature(cfeature.COASTLINE.with_scale('10m'), edgecolor='#333333', linewidth=1.0, zorder=6)
    for city, (lon, lat) in _LA_CITIES.items():
        ax.plot(lon, lat, 'k.', ms=3.5, transform=DATA_CRS, zorder=8)
        ax.text(lon + 0.012, lat + 0.012, city, fontsize=5.5, color='#222222',
                transform=DATA_CRS, zorder=9, va='bottom', ha='left',
                bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.55, ec='none'))
    gl = ax.gridlines(draw_labels=True, linewidth=0.4, color='grey',
                      alpha=0.5, linestyle='--', crs=DATA_CRS)
    gl.top_labels = gl.right_labels = False
    return ax

# ── Mean observed NO₂ per LA Basin site (for color coding) ────────────────────
mean_obs = np.nanmean(obs_all, axis=0)   # (N_la,)

fig = plt.figure(figsize=(11, 8))
ax  = _la_map_axes(fig)

sc = ax.scatter(la_lons, la_lats, c=mean_obs, cmap='YlOrRd', vmin=0, vmax=30,
                s=100, alpha=0.95, edgecolors='k', linewidths=0.5,
                transform=DATA_CRS, zorder=7)
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', shrink=0.72, pad=0.02)
cbar.set_label('Mean AirNow NO\u2082 (PPB)', fontsize=9)

# Label each site
for i, (name, lat, lon) in enumerate(zip(la_names, la_lats, la_lons)):
    ax.text(lon + 0.02, lat + 0.01, name, fontsize=6, transform=DATA_CRS,
            zorder=8, va='bottom', ha='left',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.65, ec='none'))

ax.set_title(
    'GraphMamba Inference Sites — LA Basin\n'
    'AirNow observed NO\u2082 mean (Jul–Sep 2024)',
    fontsize=11, fontweight='bold', pad=8)
plt.tight_layout()
fig.savefig(OUT_DIR / 'gm04_la_sites_map.png', dpi=150, bbox_inches='tight')
plt.show()


## 3 · Hourly Time Series — Predicted vs AirNow Observed


In [ ]:
def _site_idx(name_fragment):
    """Return local (0-based within la_names) index for a site name fragment."""
    matches = [i for i, n in enumerate(la_names)
               if name_fragment.lower() in n.lower()]
    if not matches:
        raise KeyError(f'No LA Basin site matching {name_fragment!r}')
    return matches[0]

for site_frag, color in zip(FOCUS_SITES, SITE_COLORS):
    try:
        i = _site_idx(site_frag)
    except KeyError as e:
        print(e); continue

    pred_s = pred_all[:, i]
    obs_s  = obs_all[:,  i]
    valid  = np.isfinite(obs_s)

    full_name = la_names[i]
    n_valid   = valid.sum()

    # ── metrics ────────────────────────────────────────────────────────────
    yt, yp = obs_s[valid], pred_s[valid]
    rmse   = np.sqrt(mean_squared_error(yt, yp))
    mae    = mean_absolute_error(yt, yp)
    r2     = r2_score(yt, yp)

    # ── figure: 2-row gridspec (plot + map below) ──────────────────────────
    fig = plt.figure(figsize=(14, 8))
    gs  = fig.add_gridspec(2, 3, height_ratios=[3, 2], hspace=0.40)
    ax  = fig.add_subplot(gs[0, :])
    ax_map = fig.add_subplot(gs[1, 1], projection=MAP_PROJ)

    # Observed (scatter for sparse hourly data)
    ax.scatter(times[valid], obs_s[valid], color=color, s=7, alpha=0.55,
               label='AirNow observed', zorder=4)
    # Predicted line
    ax.plot(times, pred_s, color='crimson', lw=0.8, alpha=0.8,
            label='GraphMamba predicted', zorder=5)

    ax.set_xlim(times[0], times[-1])
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.set_xlabel('Date (UTC)')
    ax.set_ylabel('NO\u2082 (PPB)')
    ax.set_title(f'Hourly NO\u2082 — {full_name}  (Jul–Sep 2024)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.text(0.01, 0.97,
            f'RMSE : {rmse:.2f} PPB\nMAE  : {mae:.2f} PPB\nR²   : {r2:.3f}\nN    : {n_valid:,}',
            transform=ax.transAxes, fontsize=8, va='top', ha='left',
            family='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.88, ec='#cccccc'))

    # ── inset map ─────────────────────────────────────────────────────────
    ax_map.set_extent([LON_MIN-.1, LON_MAX+.1, LAT_MIN-.1, LAT_MAX+.1], crs=DATA_CRS)
    ax_map.add_feature(cfeature.OCEAN.with_scale('10m'),    facecolor='#c8dff0', zorder=0)
    ax_map.add_feature(cfeature.LAND.with_scale('10m'),     facecolor='#f5f1eb', zorder=1)
    ax_map.add_feature(cfeature.LAKES.with_scale('10m'),    facecolor='#c8dff0', zorder=2)
    ax_map.add_feature(cfeature.NaturalEarthFeature('cultural', 'urban_areas', '10m',
                       facecolor='#ddd5c8', edgecolor='none'), zorder=2)
    ax_map.add_feature(cfeature.NaturalEarthFeature('cultural', 'admin_2_counties', '10m',
                       facecolor='none', edgecolor='#cccccc', linewidth=0.4), zorder=3)
    ax_map.add_feature(cfeature.NaturalEarthFeature('cultural', 'roads', '10m',
                       facecolor='none', edgecolor='#cc8844', linewidth=0.4), zorder=4)
    ax_map.add_feature(cfeature.STATES.with_scale('10m'),    edgecolor='#888888', linewidth=0.5, zorder=5)
    ax_map.add_feature(cfeature.COASTLINE.with_scale('10m'), edgecolor='#333333', linewidth=0.7, zorder=6)
    ax_map.scatter(la_lons, la_lats, color='#bbbbbb', s=15, alpha=0.6,
                   transform=DATA_CRS, zorder=7)
    ax_map.scatter([la_lons[i]], [la_lats[i]], color=color, s=110,
                   edgecolors='k', linewidths=0.7, transform=DATA_CRS, zorder=8)
    ax_map.set_title(f'{full_name} — location', fontsize=8, pad=4)

    plt.tight_layout()
    tag = site_frag.split()[0].lower()
    fig.savefig(OUT_DIR / f'gm04_ts_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 4 · Daily Aggregated Comparison


In [ ]:
def _daily_means(ts_1d, obs_1d, times):
    """Return (dates, daily_pred, daily_obs) DataFrames, only days with ≥4 valid obs."""
    df = pd.DataFrame({'pred': ts_1d, 'obs': obs_1d}, index=times)
    daily_obs  = df['obs'].resample('D').mean()
    daily_pred = df['pred'].resample('D').mean()
    # require at least 4 hourly observations per day to keep the daily mean
    obs_count  = df['obs'].resample('D').count()
    mask = obs_count >= 4
    return daily_pred[mask], daily_obs[mask]

for site_frag, color in zip(FOCUS_SITES, SITE_COLORS):
    try:
        i = _site_idx(site_frag)
    except KeyError as e:
        print(e); continue

    full_name = la_names[i]
    dpred, dobs = _daily_means(pred_all[:, i], obs_all[:, i], times)

    valid = dpred.notna() & dobs.notna()
    yt, yp = dobs[valid].values, dpred[valid].values
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)

    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(dobs.index,  dobs.values,  color=color,   lw=1.4, label='AirNow observed (daily mean)')
    ax.plot(dpred.index, dpred.values, color='crimson', lw=1.4, ls='--',
            label='GraphMamba predicted (daily mean)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily Mean NO\u2082 (PPB)')
    ax.set_title(f'Daily Mean NO\u2082 — {full_name}  (Jul–Sep 2024)',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.35)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.text(0.01, 0.97,
            f'RMSE : {rmse:.2f} PPB\nMAE  : {mae:.2f} PPB\nR²   : {r2:.3f}\nN (days) : {int(valid.sum())}',
            transform=ax.transAxes, fontsize=8, va='top', ha='left',
            family='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.88, ec='#cccccc'))
    plt.tight_layout()
    tag = site_frag.split()[0].lower()
    fig.savefig(OUT_DIR / f'gm04_daily_{tag}.png', dpi=150, bbox_inches='tight')
    plt.show()


## 5 · Scatter — Predicted vs Observed (All LA Basin Sites)


In [ ]:
# Pool all valid paired (obs, pred) values from every LA Basin site
all_obs_flat  = obs_all.ravel()
all_pred_flat = pred_all.ravel()
valid_flat = np.isfinite(all_obs_flat) & np.isfinite(all_pred_flat)
yt_all, yp_all = all_obs_flat[valid_flat], all_pred_flat[valid_flat]

rmse_all = np.sqrt(mean_squared_error(yt_all, yp_all))
mae_all  = mean_absolute_error(yt_all, yp_all)
r2_all   = r2_score(yt_all, yp_all)
lim      = max(np.nanpercentile(yt_all, 99), np.nanpercentile(yp_all, 99)) * 1.05

fig, ax = plt.subplots(figsize=(7, 7))
ax.hexbin(yt_all, yp_all, gridsize=70, cmap='YlOrRd', mincnt=1, zorder=3)
ax.plot([0, lim], [0, lim], 'k--', lw=1.2, label='1:1 line', zorder=5)
cb = plt.colorbar(ax.collections[0], ax=ax, pad=0.02)
cb.set_label('Count', fontsize=9)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('AirNow Observed NO\u2082 (PPB)')
ax.set_ylabel('GraphMamba Predicted NO\u2082 (PPB)')
ax.set_title('Predicted vs Observed — All LA Basin Sites\n(hourly, Jul–Sep 2024)',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.25)
ax.text(0.03, 0.97,
        f'RMSE : {rmse_all:.2f} PPB\nMAE  : {mae_all:.2f} PPB\nR²   : {r2_all:.3f}\nN    : {len(yt_all):,}',
        transform=ax.transAxes, fontsize=9, va='top', ha='left', family='monospace',
        bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.88, ec='#cccccc'))
plt.tight_layout()
fig.savefig(OUT_DIR / 'gm04_scatter_all_sites.png', dpi=150, bbox_inches='tight')
plt.show()


## 6 · Per-Site Metrics & Spatial Error Map


In [ ]:
# ── Per-site metrics ───────────────────────────────────────────────────────────
records = []
site_rmse = np.full(len(la_names), np.nan)
site_mae  = np.full(len(la_names), np.nan)
site_r2   = np.full(len(la_names), np.nan)

for i, name in enumerate(la_names):
    obs_s  = obs_all[:,  i]
    pred_s = pred_all[:, i]
    valid  = np.isfinite(obs_s) & np.isfinite(pred_s)
    if valid.sum() < 10:
        continue
    yt, yp = obs_s[valid], pred_s[valid]
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae  = mean_absolute_error(yt, yp)
    r2   = r2_score(yt, yp)
    site_rmse[i], site_mae[i], site_r2[i] = rmse, mae, r2
    records.append({'Site': name, 'Lat': la_lats[i], 'Lon': la_lons[i],
                    'RMSE (PPB)': round(rmse, 3), 'MAE (PPB)': round(mae, 3),
                    'R²': round(r2, 4), 'N (valid obs)': int(valid.sum())})

metrics_df = pd.DataFrame(records).sort_values('RMSE (PPB)')
metrics_df.to_csv(OUT_DIR / 'gm04_la_site_metrics.csv', index=False)
print(metrics_df.to_string(index=False))


In [ ]:
# ── Spatial RMSE map ──────────────────────────────────────────────────────────
valid_site = np.isfinite(site_rmse)

fig = plt.figure(figsize=(11, 8))
ax  = _la_map_axes(fig)

sc = ax.scatter(
    la_lons[valid_site], la_lats[valid_site],
    c=site_rmse[valid_site], cmap='RdYlGn_r', vmin=0, vmax=20,
    s=120, alpha=0.95, edgecolors='k', linewidths=0.5,
    transform=DATA_CRS, zorder=7
)
cbar = plt.colorbar(sc, ax=ax, orientation='vertical', shrink=0.72, pad=0.02)
cbar.set_label('Hourly RMSE (PPB)', fontsize=9)

for i in np.where(valid_site)[0]:
    ax.text(la_lons[i] + 0.02, la_lats[i] + 0.01,
            f'{la_names[i]}\n{site_rmse[i]:.1f}',
            fontsize=5.5, transform=DATA_CRS, zorder=8,
            va='bottom', ha='left',
            bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.65, ec='none'))

ax.set_title(
    'GraphMamba Hourly RMSE per Site — LA Basin\n(Jul–Sep 2024)',
    fontsize=11, fontweight='bold', pad=8
)
plt.tight_layout()
fig.savefig(OUT_DIR / 'gm04_la_rmse_map.png', dpi=150, bbox_inches='tight')
plt.show()
